## Step 1 — Authenticate with HuggingFace Hub

Log in with a write-scoped HuggingFace token so checkpoints can be pushed automatically during training.
Replace `YOUR_HF_TOKEN` with your own token from https://huggingface.co/settings/tokens


In [1]:
from huggingface_hub import login; login(token="YOUR_HF_TOKEN")

## Step 2 — Install Unsloth

Unsloth provides 2x faster QLoRA fine-tuning with lower VRAM usage by fusing kernels and smartly offloading gradients.
The `cu128-torch260` variant matches the CUDA 12.8 + PyTorch 2.10 environment on the GPU instance (RTX 5090, vast.ai).


In [2]:
pip install "unsloth[cu128-torch260]" --upgrade

  Using cached unsloth-2026.4.4-py3-none-any.whl.metadata (55 kB)
  Using cached unsloth_zoo-2026.4.4-py3-none-any.whl.metadata (32 kB)
Using cached unsloth-2026.4.4-py3-none-any.whl (62.6 MB)
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.4.2
    Uninstalling unsloth_zoo-2026.4.2:
      Successfully uninstalled unsloth_zoo-2026.4.2
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.4.2
    Uninstalling unsloth-2026.4.2:
      Successfully uninstalled unsloth-2026.4.2━━━━━━━━━━━━━━━━━━━ 1/2 [unsloth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [unsloth]m1/2 [unsloth]
Note: you may need to restart the kernel to use updated packages.


## Step 3 — Install Flash Attention 2

Flash Attention 2 accelerates the attention computation for long sequences.
We use `--no-build-isolation` to reuse the existing PyTorch installation.
If compilation is cancelled, Unsloth falls back to Xformers automatically with no impact on correctness.


In [2]:
pip install flash-attn --no-build-isolation --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 91.4 MB/s  0:00:00
  Preparing metadata (pyproject.toml) ... done
anceled
Note: you may need to restart the kernel to use updated packages.


## Step 4 — Imports

Core dependencies:
- `FastLanguageModel` (Unsloth): fast 4-bit model loading + LoRA patching
- `SFTTrainer` / `SFTConfig` (TRL): supervised fine-tuning loop with sequence packing
- `Dataset` (HuggingFace datasets): in-memory dataset wrapper


In [2]:
import json
import time
from pathlib import Path

from huggingface_hub import login
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Step 5 — Hyperparameter Configuration

| Parameter | Value | Rationale |
|---|---|---|
| Base model | `Qwen3-1.7B-bnb-4bit` | 1.7B params, 4-bit quantised, fits in <4 GB VRAM |
| MAX_SEQ | 8192 | Covers ~97% of ContractNLI documents without truncation |
| LORA_RANK | 4 | Low rank keeps trainable params at 0.25% (4.4M / 1.7B) |
| LR | 2e-4 | Standard for QLoRA SFT |
| ITERS | 1269 | ~7 epochs over 381 training documents with packing |
| BATCH_SIZE | 1 | Single-GPU; gradient checkpointing keeps VRAM usage flat |
| SAVE_STEPS | 120 | Checkpoint pushed to HF Hub every ~7 min at 3.6s/step |

Set `HF_REPO_ID` to your own HuggingFace repository before running.


In [3]:
MODEL = "unsloth/Qwen3-1.7B-bnb-4bit"
MAX_SEQ      = 8192                              # covers ~97% of docs
NUM_LAYERS   = 8                                # LoRA layers
LORA_RANK    = 4
LR           = 2e-4
ITERS        = 1269                                # 5% sample: 19 docs × 3 epochs
BATCH_SIZE   = 1
GRAD_ACCUM   = 1                                 # effective batch = 4
SAVE_STEPS   = 120 
OUTPUT_DIR   = "./adapters"

HF_TOKEN     = "YOUR_HF_TOKEN" # paste your HF write token here
HF_REPO_ID   = "Youssef-Malek/contractnli-vast-ai-qwen3-1.7b"    # e.g. "youssef/contractnli-qwen3-4b"

TRAIN_JSONL  = "./train.jsonl"
VAL_JSONL    = "./valid.jsonl"

## Step 6 — Training

The `main()` function runs the full fine-tuning pipeline:

1. **Load model**: downloads `Qwen3-1.7B-bnb-4bit` from HuggingFace and applies Unsloth kernel patches.
2. **Apply LoRA**: wraps all attention and MLP projection layers with low-rank adapters. Only 4.4M of 1.7B parameters are trained (0.25%).
3. **Load datasets**: reads `train.jsonl` (381 examples) and `valid.jsonl` (42 examples) — Qwen3 chat-template formatted NLI examples generated by `01_preprocess.py`.
4. **SFTConfig**: `packing=True` concatenates multiple short contracts into 8192-token sequences, doubling effective throughput. Eval packing is disabled to avoid OOM on the validation set.
5. **Fault-tolerant checkpointing**: every 120 steps the checkpoint is pushed to the HuggingFace Hub (`hub_strategy=all_checkpoints`). If the instance is interrupted, training can resume from the latest checkpoint.
6. **Training result**: completed in 1.28 hrs at 3.6s/step on an RTX 5090 (31 GB VRAM, vast.ai).


In [4]:
def load_jsonl(path: str) -> Dataset:
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return Dataset.from_list(rows)


def main():
    login(token=HF_TOKEN)

    print(f"Loading model: {MODEL}")
    t0 = time.perf_counter()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=MAX_SEQ,
        load_in_4bit=True,
        fast_inference=False,
        attn_implementation="flash_attention_2",
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_RANK * 2,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
    )

    print(f"Model loaded in {time.perf_counter()-t0:.1f}s")
    model.print_trainable_parameters()

    print("\nLoading datasets...")
    train_ds = load_jsonl(TRAIN_JSONL)
    val_ds   = load_jsonl(VAL_JSONL)
    print(f"  train={len(train_ds)}, val={len(val_ds)}")

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        max_steps=ITERS,
        learning_rate=LR,
        fp16=False,
        bf16=True,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        save_strategy="steps",
        eval_packing=False,
        per_device_eval_batch_size=1,
        save_steps=SAVE_STEPS,
        save_total_limit=None,              # keep last 3 checkpoints locally
        push_to_hub=True,
        hub_model_id=HF_REPO_ID,
        hub_token=HF_TOKEN,
        hub_strategy="all_checkpoints",       # push every checkpoint, not just best
        load_best_model_at_end=False,
        seed=42,
        dataset_text_field="text",
        packing=True,
        optim="adamw_8bit",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=training_args,
    )

    print(f"\nTraining: {ITERS} steps | max_seq={MAX_SEQ} | rank={LORA_RANK}")
    print(f"Checkpointing every {SAVE_STEPS} steps → {HF_REPO_ID}\n")

    t_train = time.perf_counter()
    trainer.train()
    elapsed = time.perf_counter() - t_train

    per_step = elapsed / ITERS
    print(f"\nDone in {elapsed/3600:.2f} hrs  |  {per_step:.1f}s / step")

    # Final push of merged adapters
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f"\nFinal adapters pushed to https://huggingface.co/{HF_REPO_ID}")


if __name__ == "__main__":
    main()


Loading model: unsloth/Qwen3-1.7B-bnb-4bit
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Saved model to https://huggingface.co/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b


HTTP Error 504 thrown while requesting POST https://huggingface.co/api/models/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b/preupload/main
[huggingface_hub.utils._http|WARNING]HTTP Error 504 thrown while requesting POST https://huggingface.co/api/models/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b/preupload/main
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Final adapters pushed to https://huggingface.co/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b


## Step 7 — Manual Checkpoint Upload

After training, the two most recent intermediate checkpoints (step 1080 and 1200) are uploaded explicitly to HuggingFace.
This ensures the full checkpoint history is preserved even if the final automatic push partially failed due to a 504 gateway timeout.


In [5]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="./adapters/checkpoint-1080",
    repo_id="Youssef-Malek/contractnli-vast-ai-qwen3-1.7b",
    path_in_repo="checkpoint-1080",
    token=HF_TOKEN,
)

api.upload_folder(
    folder_path="./adapters/checkpoint-1200",
    repo_id="Youssef-Malek/contractnli-vast-ai-qwen3-1.7b",
    path_in_repo="checkpoint-1200",
    token=HF_TOKEN,
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b/commit/32dcf63f102d56010f686033a4e310f878a91258', commit_message='Upload folder using huggingface_hub', commit_description='', oid='32dcf63f102d56010f686033a4e310f878a91258', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Youssef-Malek/contractnli-vast-ai-qwen3-1.7b', endpoint='https://huggingface.co', repo_type='model', repo_id='Youssef-Malek/contractnli-vast-ai-qwen3-1.7b'), pr_revision=None, pr_num=None)